# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset ([source](https://doi.org/10.71728/senscience.vq4a-28xa)) using the `mlcroissant` library. We will leverage the Croissant schema to discover record sets, select fields by `@id`, and perform example analyses.

### Dataset Source
The dataset is described by a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

You may need to adjust analysis examples for your specific use case, but this notebook covers the key workflow.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (and cache internally)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's examine the available record sets, and for each record set, the available field and column `@id`s.

All entities are referenced via their `@id`, as per Croissant best practices.

In [ ]:
# List all record sets by @id
if not metadata.record_sets:
    print('No record sets found in the metadata.')
else:
    print('Record Sets:')
    for rs in metadata.record_sets:
        print(f" - {rs['@id']}: {rs.get('name', '(no name)')}")
        # Show fields in this record set
        fields = rs.get('fields', [])
        print('   Fields:')
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"     - {fid}")
        # Show columns (if any file object)
        file_obj = rs.get('fileObject') or rs.get('file_object') or None
        if file_obj and isinstance(file_obj, dict) and 'columns' in file_obj:
            print('   Columns:')
            for col in file_obj['columns']:
                cid = col['@id'] if isinstance(col, dict) else col
                print(f"     - {cid}")

#### Example: Print records from a record set

Replace `<record_set_id>` below with one of the `@id`s listed above for further inspection.

In [ ]:
# Example: iterate over records in a record set
all_record_set_ids = [rs['@id'] for rs in (metadata.record_sets or [])]
if all_record_set_ids:
    record_set_id = all_record_set_ids[0]  # take the first as default
    print(f'Sample records from record set: {record_set_id}')
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint.pprint(record)
        if i >= 2:
            break
else:
    print('No record sets available to show records.')

## 3. Data Extraction
We load each record set into a pandas DataFrame for convenient exploration. Each DataFrame is keyed by its record set `@id`.

**Note:** All field and column references use their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in (metadata.record_sets or [])]
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set: {rsid}, {len(records)} records.")

# Show columns in the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Fields/columns in '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Here, we demonstrate filtering on a numeric field (using the field or column `@id`), normalizing, and grouping. You should substitute `<numeric_field_id>` and `<group_field_id>` with the relevant field/column `@id`s from your record set.

Example below uses the first available record set and tries to auto-select a likely numeric field.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Select the first record set with any numeric fields
chosen_rs = None
numeric_field_id = None
group_field_id = None
for rsid in record_set_ids:
    df = dataframes[rsid]
    numeric_candidates = [c for c in df.columns if is_numeric_dtype(df[c])]
    if numeric_candidates:
        chosen_rs = rsid
        numeric_field_id = numeric_candidates[0]
        # Try to pick a non-numeric column for grouping
        non_num_candidates = [c for c in df.columns if not is_numeric_dtype(df[c])]
        group_field_id = non_num_candidates[0] if non_num_candidates else None
        break
if not chosen_rs:
    print('No numeric fields found in any record set.')
else:
    df = dataframes[chosen_rs]
    print(f"Using record set: {chosen_rs}")
    print(f"Numeric field: {numeric_field_id}")
    if group_field_id:
        print(f"Group-by field: {group_field_id}")
    else:
        print('No group field identified.')
    # Filter
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in ['i', 'f'] else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field for our example record set.

You can adjust the plot to other fields/groupings as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[chosen_rs][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {chosen_rs}")
    plt.xlabel(numeric_field_id)
    plt.show()
else:
    print('No numeric field available to plot.')

## 6. Conclusion
In this notebook, we:
- Loaded and described the FAIR² mass spectrometry dataset schema using `mlcroissant` and pandas
- Explored available record sets, fields, and columns using their `@id`s
- Loaded records from a selected record set and performed basic EDA (filter, normalize, group, visualize)

You can adapt this workflow for more advanced analysis, leveraging the structured Croissant schema for robust field/entity dereferencing.